# P1 · Block 2 — Bronze → Silver

- **Source:** `retail.bronze.orders`, `retail.bronze.customers`
- **Target:** `retail.silver.orders`, `retail.silver.orders_quarantine`, `retail.silver.dim_customer`
- **Gold:** aggregates live in `03_gold` — separate notebook, separate task, so a broken rollup never forces a re-run of the merge underneath it.

### Silver contract

Bronze is an as-landed mirror, defects included. Silver guarantees one row per order, latest
state, one status vocabulary, and a `dim_customer` that keeps history rather than overwriting it.

Rows failing a quality rule are **quarantined, not dropped** — a discarded row is a row you
cannot explain when someone asks why revenue moved.

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

STORAGE = "stgaccdeprep"
CATALOG = "retail"
spark.sql(f"USE CATALOG {CATALOG}")

RAW = f"abfss://raw@{STORAGE}.dfs.core.windows.net"

BRONZE_ORDERS    = "retail.bronze.orders"
BRONZE_CUSTOMERS = "retail.bronze.customers"
SILVER_ORDERS    = "retail.silver.orders"
QUARANTINE       = "retail.silver.orders_quarantine"
DIM_CUSTOMER     = "retail.silver.dim_customer"

bronze = spark.table(BRONZE_ORDERS)
print(f"bronze.orders : {bronze.count()} rows")

### Deduplication is a ranking problem, not a uniqueness problem

The 40 restatements share an `order_id` but differ in `updated_ts` and `status`, so every row
is unique and `DISTINCT` removes nothing. Silver needs the *latest* row per key.

In [0]:
print("total rows         :", bronze.count())
print("DISTINCT whole-row :", bronze.distinct().count())
print("DISTINCT order_id  :", bronze.select("order_id").distinct().count())

### Is the ordering deterministic?

`row_number` over an ambiguous `ORDER BY` picks a different winner on different runs, which
breaks idempotency downstream.

`_ingest_ts` looks like a tie-breaker and is not — `current_timestamp()` is evaluated once per
query, so the whole batch carries one value, on every run, forever. `_source_file` also reads 1
today, but only because one file has landed; it is evaluated per row and will discriminate once
Block 3 adds more.

Two audit columns that look alike, different granularity. Checked rather than assumed.

In [0]:
ties = (bronze.groupBy("order_id", "updated_ts")
              .count().filter(F.col("count") > 1).count())

print("order_id + updated_ts collisions :", ties)
print("distinct _ingest_ts values       :", bronze.select("_ingest_ts").distinct().count())
print("distinct _source_file values     :", bronze.select("_source_file").distinct().count())

### Dedup to latest state

`ROW_NUMBER` partitioned by `order_id`, ordered by `updated_ts` descending, rank 1 wins.
640 → 600.

`_corrupt_record` is dropped only after asserting it is empty. Dropping a column you have not
checked is how silent data loss enters a pipeline.

In [0]:
corrupt = bronze.filter(F.col("_corrupt_record").isNotNull()).count()
assert corrupt == 0, f"{corrupt} corrupt rows present — handle before dropping the column"

w_latest = Window.partitionBy("order_id").orderBy(F.col("updated_ts").desc())

deduped = (bronze
    .withColumn("_rn", F.row_number().over(w_latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_corrupt_record"))

print("before:", bronze.count(), "| after:", deduped.count(), "| removed:", bronze.count() - deduped.count())

### One status vocabulary

12 distinct values collapse to 4 under `TRIM` then `UPPER`. Values are bracketed in the output
because leading and trailing whitespace is invisible in a result grid — ` PLACED` and `PLACED`
look identical until you box them.

Both tables run on the same 640 rows so the pair reads as one transform.

Trade-off: normalising here means the original casing survives only in Bronze. Accepted —
Bronze is the audit copy, and nothing downstream needs the variants.

In [0]:
print("BEFORE — raw bronze status values")
display(
    bronze.groupBy("status").count()
          .withColumn("status", F.concat(F.lit("["), F.col("status"), F.lit("]")))
          .orderBy(F.desc("count"))
)

In [0]:
status_clean = F.upper(F.trim(F.col("status")))

print("AFTER — trimmed and upper-cased (same 640 rows)")
display(
    bronze.withColumn("status", status_clean)
          .groupBy("status").count().orderBy(F.desc("count"))
)

print("distinct status:",
      bronze.select("status").distinct().count(), "->",
      bronze.select(status_clean).distinct().count())

# pipeline step — applied to the deduped 600
cleaned = deduped.withColumn("status", status_clean)

### Three failure classes, one quarantine

Null amount, negative amount, and a `customer_id` with no matching customer. A row can fail
more than one, so reasons accumulate in an array rather than a single column — you cannot route
a row twice.

The customer lookup is `.distinct()` before the join. A dimension join that is not unique on its
key silently multiplies the fact table, and the row count still looks plausible enough to ship.

In [0]:
customer_ids = (spark.table(BRONZE_CUSTOMERS)
                     .select("customer_id").distinct()
                     .withColumn("_customer_exists", F.lit(True)))

flagged = (cleaned
    .join(customer_ids, on="customer_id", how="left")
    .withColumn("_reasons", F.array_compact(F.array(
        F.when(F.col("amount").isNull(),           F.lit("NULL_AMOUNT")),
        F.when(F.col("amount") < 0,                F.lit("NEGATIVE_AMOUNT")),
        F.when(F.col("_customer_exists").isNull(), F.lit("ORPHAN_CUSTOMER")),
    )))
    .drop("_customer_exists"))

display(flagged.groupBy("_reasons").count().orderBy(F.desc("count")))

### Split, and reconcile

Clean plus quarantined must equal the deduped input. An unreconciled split is a pipeline that
loses rows without saying so.

These counts do not match the Bronze defect counts, and should not: dedup ran first, so five
defective rows that a later restatement superseded were gone before the rules ever ran.

One row carries two reasons — 43 violations across 42 rows. Reconcile on rows, never on the
sum of a reason column.

In [0]:
silver_ok   = flagged.filter(F.size("_reasons") == 0).drop("_reasons")
quarantined = flagged.filter(F.size("_reasons") >  0)

print("clean       :", silver_ok.count())
print("quarantined :", quarantined.count())
print("reconciles  :", silver_ok.count() + quarantined.count(), "(expect 600)")

### Quarantine is a table, not a log line

Written with `overwrite` because it is a full recompute of the same input, not an accumulating
journal. Re-running the notebook must not double it.

`_quarantined_at` is the only column added here — when a row is asked about, the first question
is always when it was caught.

In [0]:
(quarantined
    .withColumn("_quarantined_at", F.current_timestamp())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE))

print("quarantine rows:", spark.table(QUARANTINE).count())
display(spark.table(QUARANTINE).select("order_id","customer_id","amount","status","_reasons"))

### Create empty, then merge — even on first load

The table is created with the right schema and no rows, so the very first load goes through the
same `MERGE` path as every load after it. A pipeline whose first run takes a different code path
than its second is a pipeline with an untested branch in it.

In [0]:
if not spark.catalog.tableExists(SILVER_ORDERS):
    (silver_ok.limit(0)
        .write.format("delta")
        .saveAsTable(SILVER_ORDERS))

print("silver.orders rows:", spark.table(SILVER_ORDERS).count())

### Idempotency

`WHEN MATCHED AND s.updated_ts > t.updated_ts` is the load-bearing clause. Without it every
re-run rewrites all 558 rows — correct output, wasted I/O, and a Delta history full of commits
that changed nothing.

Run 1 inserts. Run 2 must report zero affected rows and leave the count untouched. That is the
difference between "the numbers look right" and "re-running is safe".

In [0]:
silver_ok.createOrReplaceTempView("silver_orders_src")

MERGE_SQL = """
MERGE INTO retail.silver.orders AS t
USING silver_orders_src AS s
   ON t.order_id = s.order_id
 WHEN MATCHED AND s.updated_ts > t.updated_ts THEN UPDATE SET *
 WHEN NOT MATCHED THEN INSERT *
"""

print("--- run 1 ---")
display(spark.sql(MERGE_SQL))
print("rows after run 1:", spark.table(SILVER_ORDERS).count())

print("--- run 2 ---")
display(spark.sql(MERGE_SQL))
print("rows after run 2:", spark.table(SILVER_ORDERS).count())

### SCD Type 2 — compare attributes, not timestamps

A changed `updated_ts` is not a change. The source carries two rows whose timestamp moved while
every attribute stayed identical; versioning on the timestamp would create two rows of history
that record nothing.

Comparison is a SHA-256 over the tracked attributes only. Each column is coalesced to a sentinel
first because `concat_ws` skips nulls — without it, `("A", null, "B")` and `("A", "B", null)`
hash the same.

In [0]:
customers_schema = StructType([
    StructField("customer_id",   StringType(),    True),
    StructField("customer_name", StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("segment",       StringType(),    True),
    StructField("country",       StringType(),    True),
    StructField("updated_ts",    TimestampType(), True),
    StructField("_corrupt_record", StringType(),  True),
])

TRACKED = ["customer_name", "city", "segment", "country"]

attr_hash = F.sha2(
    F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("<NULL>")) for c in TRACKED]), 256
)

# --- initial dimension from v1 (already in bronze) ---
dim_init = (spark.table(BRONZE_CUSTOMERS)
    .select("customer_id", *TRACKED, "updated_ts")
    .withColumn("_attr_hash", attr_hash)
    .withColumn("valid_from", F.col("updated_ts"))
    .withColumn("valid_to",   F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .drop("updated_ts"))

if not spark.catalog.tableExists(DIM_CUSTOMER):
    (dim_init.write.format("delta").saveAsTable(DIM_CUSTOMER))

print("dim_customer rows:", spark.table(DIM_CUSTOMER).count())

### What a naive implementation would have got wrong

Two counts, same source. The timestamp test versions 12 customers; the attribute hash versions
10. The gap is the two rows whose `updated_ts` moved with no attribute change — spurious history
that would sit in the dimension forever, and that nothing downstream could distinguish from a
real change.

In [0]:
v2 = (spark.read.format("csv").option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(customers_schema)
        .load(f"{RAW}/customers/customers_v2.csv")
        .select("customer_id", *TRACKED, "updated_ts")
        .withColumn("_attr_hash", attr_hash))

cur = (spark.table(DIM_CUSTOMER).filter(F.col("is_current"))
        .select("customer_id",
                F.col("_attr_hash").alias("_cur_hash"),
                F.col("valid_from").alias("_cur_from")))

joined = v2.join(cur, on="customer_id", how="left")

new_customers = joined.filter(F.col("_cur_hash").isNull())
changed       = joined.filter(F.col("_cur_hash").isNotNull() &
                              (F.col("_attr_hash") != F.col("_cur_hash")))
unchanged     = joined.filter(F.col("_attr_hash") == F.col("_cur_hash"))

naive = joined.filter(F.col("_cur_from").isNotNull() &
                      (F.col("updated_ts") > F.col("_cur_from")))

print("v2 rows              :", v2.count())
print("new customers        :", new_customers.count())
print("changed (hash)       :", changed.count())
print("unchanged            :", unchanged.count())
print()
print("would version on ts  :", naive.count(), "  <-- naive")
print("versions on attrs    :", changed.count(), "  <-- correct")

### Two rows per change, one merge

A single `MERGE` cannot both close a row and insert its replacement — the same target row would
have to match twice. The staging set carries each change twice: once keyed on `customer_id` to
close the current version, once with a null key that can never match and therefore inserts.

`ON t.customer_id = s._merge_key AND t.is_current` scopes the close to the open row only.
Without the `is_current` predicate the merge matches every historical version and reopens them.

In [0]:
to_close  = changed.withColumn("_merge_key", F.col("customer_id"))
to_insert = (changed.unionByName(new_customers)
                    .withColumn("_merge_key", F.lit(None).cast("string")))

to_close.unionByName(to_insert).createOrReplaceTempView("dim_customer_stage")

spark.sql("""
MERGE INTO retail.silver.dim_customer AS t
USING dim_customer_stage AS s
   ON t.customer_id = s._merge_key AND t.is_current = true
 WHEN MATCHED THEN UPDATE SET
       t.is_current = false,
       t.valid_to   = s.updated_ts
 WHEN NOT MATCHED THEN INSERT
       (customer_id, customer_name, city, segment, country,
        _attr_hash, valid_from, valid_to, is_current)
 VALUES (s.customer_id, s.customer_name, s.city, s.segment, s.country,
        s._attr_hash, s.updated_ts, NULL, true)
""").display()

### The only assertion that matters

A Type 2 dimension is broken the moment one key has two open rows — every downstream join
silently doubles. Row totals can look fine while this is wrong, so it is checked explicitly.

In [0]:
d = spark.table(DIM_CUSTOMER)

print("total rows        :", d.count())                             # 75
print("current           :", d.filter("is_current").count())        # 65
print("closed            :", d.filter("not is_current").count())    # 10
print("distinct customers:", d.select("customer_id").distinct().count())  # 65

dupes = (d.filter("is_current").groupBy("customer_id")
          .count().filter("count > 1").count())
print("keys with >1 open row:", dupes, "(must be 0)")

display(d.filter(F.col("customer_id").isin(
    [r.customer_id for r in changed.select("customer_id").limit(3).collect()]
)).orderBy("customer_id", "valid_from"))